# SMS Spam Detection (10-Fold Cross Validation)

**Deployment Link:** [https://spam-detector-hassan.streamlit.app](https://spam-detector-hassan.streamlit.app)

**Name:** Muhammad Hassan Yousaf  
**Roll Number:** FA24-BSE-082  
**Course:** Machine Learning Fundamentals (AIC354), Assignment 4, Task 2  
**Instructor:** Dr. Rao Muhammad Adeel Nawab  
**University:** COMSATS University Islamabad, Lahore Campus  
**GitHub:** [https://github.com/MHassan0000/ML-Prediction-Models](https://github.com/MHassan0000/ML-Prediction-Models)

For this assignment I picked SMS spam detection as my machine learning problem. The goal is to classify each SMS message as spam or ham (legitimate). This is a good fit for the unstructured text requirement since the input is raw message text.

**Dataset:** UCI SMS Spam Collection (5,572 messages, 86.6% ham and 13.4% spam)  
**Feature Extraction:** TF-IDF Vectorizer (word unigrams, 5000 features)  
**Evaluation:** 10-Fold Cross Validation, Accuracy metric

**6 Algorithms compared:**
1. Logistic Regression
2. Multinomial Naive Bayes
3. Random Forest
4. Linear SVC
5. k-Nearest Neighbors
6. Gradient Boosting

To run on Google Colab: upload this notebook, then upload `spam_dataset.csv` when Step 2 prompts for it.

## Step 1: Import Libraries

In [ ]:
import os
import re
import pickle
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import sklearn
import matplotlib
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier

from tqdm import tqdm

pd.set_option('display.float_format', lambda v: f'{v:.4f}')

print(f'NumPy        : {np.__version__}')
print(f'Pandas       : {pd.__version__}')
print(f'Scikit-learn : {sklearn.__version__}')
print(f'Matplotlib   : {matplotlib.__version__}')

## Step 2: Load Dataset

In [ ]:
DATA_FILE = 'spam_dataset.csv'

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not os.path.exists(DATA_FILE):
    print(f'Upload {DATA_FILE} when the dialog appears...')
    colab_files.upload()

df = pd.read_csv(DATA_FILE)
print(f'Loaded {len(df):,} messages')
print(f'Shape: {df.shape}')
df.head(10)

## Step 3: Understand the Data

In [ ]:
class_counts = df['label'].value_counts()
class_pct    = df['label'].value_counts(normalize=True) * 100

print('Class distribution:')
for label in class_counts.index:
    print(f'  {label:>5}: {class_counts[label]:>5,} messages ({class_pct[label]:.1f}%)')
print(f'\n  Imbalance ratio: {class_counts["ham"] / class_counts["spam"]:.1f} ham per spam message')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#2ecc71', '#e74c3c']
bars = axes[0].bar(class_counts.index, class_counts.values, color=colors, edgecolor='white', width=0.5)
for bar, count, pct in zip(bars, class_counts.values, class_pct.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
                 f'{count:,}\n({pct:.1f}%)', ha='center', fontweight='bold')
axes[0].set_title('SMS Class Distribution')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Number of Messages')
axes[0].spines[['top', 'right']].set_visible(False)

df['text_length'] = df['text'].apply(len)
for label, color in zip(['ham', 'spam'], colors):
    axes[1].hist(df[df['label'] == label]['text_length'], bins=50,
                 alpha=0.7, label=label, color=color, edgecolor='white')
axes[1].set_title('Text Length Distribution')
axes[1].set_xlabel('Number of characters')
axes[1].set_ylabel('Frequency')
axes[1].legend()
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
print('Text length statistics:')
for label in ['ham', 'spam']:
    s = df[df['label'] == label]
    print(f'\n  {label.upper()} messages:')
    print(f'    Avg characters: {s["text_length"].mean():.1f}')
    print(f'    Max characters: {s["text_length"].max()}')
    print(f'    Min characters: {s["text_length"].min()}')

In [ ]:
print('Sample HAM messages:')
for _, row in df[df['label'] == 'ham'].head(3).iterrows():
    print(f'  "{row["text"][:90]}"')

print('\nSample SPAM messages:')
for _, row in df[df['label'] == 'spam'].head(3).iterrows():
    print(f'  "{row["text"][:90]}"')

## Step 4: Pre-processing

I apply the same cleaning steps to every message: lowercase, remove URLs and numbers, strip special characters, and collapse whitespace. Spam messages often contain things like phone numbers and URLs which are removed at this step so the classifier focuses on word patterns instead.

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print('Before and after cleaning:')
for idx in [0, 2, 5]:
    orig    = df.loc[idx, 'text']
    cleaned = clean_text(orig)
    label   = df.loc[idx, 'label'].upper()
    print(f'\n  [{label}] BEFORE: "{orig[:80]}"')
    print(f'          AFTER : "{cleaned[:80]}"')

df['clean_text'] = df['text'].apply(clean_text)
print(f'\nCleaned {len(df):,} messages.')

In [ ]:
le = LabelEncoder()
df['label_enc'] = le.fit_transform(df['label'])
print('Label encoding:')
for i, lbl in enumerate(le.classes_):
    print(f'  {lbl} -> {i}')

## Step 5: Feature Extraction

TF-IDF converts each message into a vector of 5000 word importance scores. Words like FREE and WIN get high scores in spam messages while ordinary conversation words score lower.

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 1))

X = tfidf.fit_transform(df['clean_text'])
y = df['label_enc'].values

print(f'TF-IDF matrix shape: {X.shape}')
print(f'Vocabulary size    : {len(tfidf.vocabulary_):,} words')
print(f'Non-zero values    : {X.nnz:,}')
print(f'Sparsity           : {(1 - X.nnz / (X.shape[0] * X.shape[1])) * 100:.2f}%')

In [ ]:
feature_names = tfidf.get_feature_names_out()
mean_scores   = X.mean(axis=0).A1
top_indices   = mean_scores.argsort()[::-1][:20]

print('Top 20 features by mean TF-IDF score:')
for rank, idx in enumerate(top_indices, 1):
    print(f'  {rank:>2}. {feature_names[idx]:<15} {mean_scores[idx]:.6f}')

## Step 6: Define 6 ML Algorithms

In [ ]:
algorithms = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Multinomial NB':      MultinomialNB(),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Linear SVC':          LinearSVC(max_iter=2000, random_state=42),
    'KNN':                 KNeighborsClassifier(n_neighbors=5),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
}

print(f'{len(algorithms)} algorithms defined:')
for i, name in enumerate(algorithms, 1):
    print(f'  {i}. {name}')

## Step 7: Set Up 10-Fold Cross Validation

In [ ]:
K  = 10
kf = KFold(n_splits=K, shuffle=True, random_state=42)

print(f'K = {K} folds | Total messages: {X.shape[0]:,}')
print(f'Training per fold : ~{X.shape[0] - X.shape[0] // K:,}')
print(f'Test per fold     : ~{X.shape[0] // K:,}')

print(f'\n  {"Fold":<6} {"Train":>8} {"Test":>8}')
for i, (tr, te) in enumerate(kf.split(X), 1):
    print(f'  {i:<6} {len(tr):>8,} {len(te):>8,}')

## Step 8: Run 10-Fold Cross Validation

In [ ]:
results              = {}
first_fold_preds     = {}
total_tasks          = len(algorithms) * K

print(f'Running {K}-fold CV for {len(algorithms)} algorithms ({total_tasks} total fits)...')

with tqdm(total=total_tasks, desc='Cross Validation', unit='fold') as pbar:
    for algo_name, algo in algorithms.items():
        fold_accs = []
        for fold_i, (tr, te) in enumerate(kf.split(X), 1):
            m = clone(algo)
            m.fit(X[tr], y[tr])
            y_pred = m.predict(X[te])
            fold_accs.append(accuracy_score(y[te], y_pred))
            if fold_i == 1:
                first_fold_preds[algo_name] = (y[te], y_pred)
            pbar.update(1)

        results[algo_name] = fold_accs
        print(f'  {algo_name:<22} -> {np.mean(fold_accs)*100:.2f}% ± {np.std(fold_accs)*100:.2f}%')

print('\nDone!')

## Step 9: Compare Results

### Per-Fold Accuracy

In [ ]:
fold_df = pd.DataFrame(results, index=[f'Fold {i}' for i in range(1, K + 1)]).T
print('Per-fold accuracy (%) for each algorithm:')
(fold_df * 100).round(2)

### Summary Table

In [ ]:
summary = pd.DataFrame({
    'Mean Accuracy (%)': [np.mean(v) * 100 for v in results.values()],
    'Std Dev (%)':       [np.std(v)  * 100 for v in results.values()],
    'Min Accuracy (%)':  [np.min(v)  * 100 for v in results.values()],
    'Max Accuracy (%)':  [np.max(v)  * 100 for v in results.values()],
}, index=results.keys()).sort_values('Mean Accuracy (%)', ascending=False)

summary.insert(0, 'Rank', range(1, len(summary) + 1))
print('Algorithm comparison sorted by mean accuracy:')
summary

### Charts

In [ ]:
COLORS       = ['#2ecc71', '#3498db', '#9b59b6', '#e67e22', '#e74c3c', '#1abc9c']
sorted_algos = summary.index.tolist()
sorted_means = [np.mean(results[a]) * 100 for a in sorted_algos]
sorted_stds  = [np.std(results[a])  * 100 for a in sorted_algos]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

bars = axes[0].barh(sorted_algos, sorted_means, xerr=sorted_stds,
                    color=COLORS, edgecolor='white', capsize=4, height=0.6)
axes[0].set_yticks(range(len(sorted_algos)))
axes[0].set_yticklabels(sorted_algos)
axes[0].set_xlabel('Accuracy (%)')
axes[0].set_title('Mean Accuracy with Std Dev')
axes[0].invert_yaxis()
axes[0].spines[['top', 'right']].set_visible(False)
for i, (mean, std) in enumerate(zip(sorted_means, sorted_stds)):
    axes[0].text(mean + std + 0.3, i, f'{mean:.2f}%', va='center', fontsize=9, fontweight='bold')

box_data = [np.array(results[a]) * 100 for a in sorted_algos]
bp = axes[1].boxplot(box_data, vert=True, patch_artist=True, labels=sorted_algos)
for patch, c in zip(bp['boxes'], COLORS):
    patch.set_facecolor(c)
    patch.set_alpha(0.7)
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy Distribution')
axes[1].tick_params(axis='x', rotation=45)
axes[1].spines[['top', 'right']].set_visible(False)

for i, algo in enumerate(sorted_algos):
    axes[2].plot(range(1, K + 1), np.array(results[algo]) * 100, marker='o',
                 label=algo, color=COLORS[i], linewidth=2)
axes[2].set_xlabel('Fold')
axes[2].set_ylabel('Accuracy (%)')
axes[2].set_title('Accuracy Across Folds')
axes[2].legend(fontsize=8)
axes[2].set_xticks(range(1, K + 1))
axes[2].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

## Step 10: Observations

In [ ]:
mean_accs  = {n: np.mean(v) * 100 for n, v in results.items()}
best_algo  = max(mean_accs, key=mean_accs.get)
worst_algo = min(mean_accs, key=mean_accs.get)

print(f'Best  algorithm : {best_algo}')
print(f'  Mean Accuracy : {mean_accs[best_algo]:.2f}%')
print(f'  Std Deviation : {np.std(results[best_algo]) * 100:.2f}%')
print()
print(f'Worst algorithm : {worst_algo}')
print(f'  Mean Accuracy : {mean_accs[worst_algo]:.2f}%')
print()
print(f'Accuracy gap    : {mean_accs[best_algo] - mean_accs[worst_algo]:.2f}%')
print()

y_test_b, y_pred_b = first_fold_preds[best_algo]
print(f'Classification Report for {best_algo} (Fold 1):')
print(classification_report(y_test_b, y_pred_b, target_names=le.classes_, digits=4))

## Step 11: Train Final Model

Train the best algorithm on all data and save the model and vectorizer.

In [ ]:
print(f'Training final model: {best_algo}')

final_model = clone(algorithms[best_algo])
final_model.fit(X, y)

print(f'Trained on {X.shape[0]:,} messages with {X.shape[1]:,} features.')

with open('spam_best_model.pkl', 'wb') as f:
    pickle.dump(final_model, f)

with open('spam_vectorizer.pkl', 'wb') as f:
    pickle.dump({'vectorizer': tfidf, 'label_encoder': le}, f)

print(f'Saved spam_best_model.pkl  ({os.path.getsize("spam_best_model.pkl")/1024:.1f} KB)')
print(f'Saved spam_vectorizer.pkl  ({os.path.getsize("spam_vectorizer.pkl")/1024:.1f} KB)')

## Step 12: Test Prediction

In [ ]:
with open('spam_best_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)
with open('spam_vectorizer.pkl', 'rb') as f:
    vec_data = pickle.load(f)
    loaded_vec = vec_data['vectorizer']
    loaded_le  = vec_data['label_encoder']

print('Model loaded successfully!')

test_messages = [
    "Congratulations! You've won a $1000 gift card. Call now!",
    "Hey, are you coming to the party tonight?",
    "URGENT: Your account has been compromised. Click here immediately",
    "Can you pick up some milk on the way home?",
    "FREE entry in a weekly competition. Text WIN to 80085 now!",
    "I'll call you later when I'm free. See you!",
]

print(f'\nTest predictions ({best_algo}):')
print('=' * 65)
for msg in test_messages:
    cleaned = clean_text(msg)
    pred    = loaded_model.predict(loaded_vec.transform([cleaned]))[0]
    label   = loaded_le.inverse_transform([pred])[0]
    icon    = 'SPAM' if label == 'spam' else 'HAM '
    print(f'  [{icon}]  "{msg}"')

## Step 13: Ready for Deployment

In [ ]:
print('SMS Spam Detection  -  Project Summary')
print('=' * 50)
print(f'  Problem         : SMS Spam Detection (Binary Text Classification)')
print(f'  Dataset         : UCI SMS Spam Collection ({len(df):,} messages)')
print(f'  Feature method  : TF-IDF Vectorizer (max_features=5000)')
print(f'  Evaluation      : {K}-Fold Cross Validation')
print(f'  Best algorithm  : {best_algo}')
print(f'  CV Accuracy     : {mean_accs[best_algo]:.2f}%')
print(f'  Saved model     : spam_best_model.pkl')
print(f'  Saved vectorizer: spam_vectorizer.pkl')
print()
print(f'  Deployment link : https://spam-detector-hassan.streamlit.app')
print(f'  GitHub          : https://github.com/MHassan0000/ML-Prediction-Models')